# RAC Weights — Plain-Text Inference

Evaluates RAC-trained classifiers without retrieval at inference time.

## 1. Imports

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..').resolve()))

## 2. Configuration

Choose models, index types and datasets to evaluate.

In [2]:
# Define paths
ROOT_DIR        = Path('../..')
WEIGHTS_RAC_DIR = ROOT_DIR / 'weigths' / 'weights_rac_best_hyperparameters'

MAX_LENGTH = 256
BATCH_SIZE = 32

# === What to evaluate — edit these lists ===
# What to evaluate — edit these lines
SELECTED_MODELS      = ['bert', 'roberta']             # 'bert' | 'hatebert' | 'roberta'
SELECTED_INDEX_TYPES = ['example', 'knowledge', 'full']
SELECTED_DATASETS    = ['IHC', 'ISHate', 'Vicomtech']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device           : {device}')
print(f'Models           : {SELECTED_MODELS}')
print(f'Index types      : {SELECTED_INDEX_TYPES}')
print(f'Datasets         : {SELECTED_DATASETS}')

Device           : cuda
Models           : ['bert', 'roberta']
Index types      : ['example', 'knowledge', 'full']
Datasets         : ['IHC', 'ISHate', 'Vicomtech']


## 3. Load Datasets

Only datasets in `SELECTED_DATASETS` are loaded.

In [3]:
from data_loaders import load_ihc_implicit_only, load_ishate_binary, load_vicomtech

DATASETS = {}
if 'IHC' in SELECTED_DATASETS:
    test_ihc = load_ihc_implicit_only(seed=42)
    DATASETS['IHC'] = {'test': test_ihc, 'text_col': 'post'}
    print(f'IHC       — test: {len(test_ihc):,}')

if 'ISHate' in SELECTED_DATASETS:
    _, test_ishate = load_ishate_binary()
    DATASETS['ISHate'] = {'test': test_ishate, 'text_col': 'text'}
    print(f'ISHate    — test: {len(test_ishate):,}')

if 'Vicomtech' in SELECTED_DATASETS:
    test_vicomtech = load_vicomtech(split='test')
    DATASETS['Vicomtech'] = {'test': test_vicomtech, 'text_col': 'text'}
    print(f'Vicomtech — test: {len(test_vicomtech):,}')

README.md:   0%|          | 0.00/792 [00:00<?, ?B/s]

implicit_hate_v1_stg1_posts.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21480 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2148 [00:00<?, ? examples/s]

Map:   0%|          | 0/2028 [00:00<?, ? examples/s]

...

Generating test split:   0%|          | 0/4368 [00:00<?, ? examples/s]

Map:   0%|          | 0/55023 [00:00<?, ? examples/s]

Map:   0%|          | 0/4368 [00:00<?, ? examples/s]

ISHate    — test: 4,368


Vicomtech — test: 478


## 4. Model Registry

Builds all (model, index_type, dataset) combinations to evaluate.

In [4]:
# Build the list of all (model, index_type, dataset) combinations
MODELS_CONFIG = [
    {
        'path':       WEIGHTS_RAC_DIR / model / 'sbert' / index_type / dataset,
        'label':      f'{model} / sbert / {index_type} / {dataset}',
        'model_name': model,
        'index_type': index_type,
        'dataset':    dataset,
        'text_col':   DATASETS[dataset]['text_col'],
    }
    for model in SELECTED_MODELS
    for index_type in SELECTED_INDEX_TYPES
    for dataset in SELECTED_DATASETS
]

print(f"{'Model / Index / Dataset':<45} Weights?")
print('-' * 55)
for m in MODELS_CONFIG:
    has = (m['path'] / 'model.safetensors').exists() or (m['path'] / 'pytorch_model.bin').exists()
    print(f"{m['label']:<45} {'✓' if has else '✗  (missing)'}")

Model / Index / Dataset                       Weights?
-------------------------------------------------------
bert / sbert / example / IHC                  ✓
bert / sbert / example / ISHate               ✓
bert / sbert / example / Vicomtech            ✓
bert / sbert / knowledge / IHC                ✓
bert / sbert / knowledge / ISHate             ✓
bert / sbert / knowledge / Vicomtech          ✓
bert / sbert / full / IHC                     ✓
bert / sbert / full / ISHate                  ✓
bert / sbert / full / Vicomtech               ✓
roberta / sbert / example / IHC               ✓
roberta / sbert / example / ISHate            ✓
roberta / sbert / example / Vicomtech         ✓
roberta / sbert / knowledge / IHC             ✓
roberta / sbert / knowledge / ISHate          ✓
roberta / sbert / knowledge / Vicomtech       ✓
roberta / sbert / full / IHC                  ✓
roberta / sbert / full / ISHate               ✓
roberta / sbert / full / Vicomtech            ✓


## 5. Helpers

In [5]:
from training_utils import compute_metrics, tokenize_plain

## 6. Evaluation Loop

For each available model: tokenize plain text (no retrieval) → predict → store metrics.

In [6]:
results = {}

# Set up a minimal Trainer just for inference
eval_args = TrainingArguments(
    output_dir='./tmp_eval',
    per_device_eval_batch_size=BATCH_SIZE,
    report_to='none',
)

# Evaluate each model
for entry in MODELS_CONFIG:
    has_weights = (entry['path'] / 'model.safetensors').exists() or (entry['path'] / 'pytorch_model.bin').exists()
    if not has_weights:
        print(f"[skip] {entry['label']} — no weights on disk")
        continue

    print(f"\n{'='*60}")
    print(f"{entry['label']}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(entry['path'])
    test_ds   = DATASETS[entry['dataset']]['test']
    tok_test  = tokenize_plain(test_ds, tokenizer, entry['text_col'])

    model   = AutoModelForSequenceClassification.from_pretrained(entry['path'])
    trainer = Trainer(model=model, args=eval_args, compute_metrics=compute_metrics)

    preds_out = trainer.predict(tok_test)
    preds     = np.argmax(preds_out.predictions, axis=-1)
    labels    = list(test_ds['label'])

    print(classification_report(labels, preds, target_names=['Non-HS', 'HS']))

    results[entry['label']] = {
        'model':      entry['model_name'],
        'index_type': entry['index_type'],
        'dataset':    entry['dataset'],
        'macro_f1':   f1_score(labels, preds, average='macro',  zero_division=0),
        'macro_p':    precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':    recall_score(labels, preds, average='macro',    zero_division=0),
    }

    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()


bert / sbert / example / IHC


              precision    recall  f1-score   support

      Non-HS       0.81      0.88      0.84      1330
          HS       0.72      0.60      0.66       698

    accuracy                           0.78      2028
   macro avg       0.77      0.74      0.75      2028
weighted avg       0.78      0.78      0.78      2028


bert / sbert / example / ISHate


              precision    recall  f1-score   support

      Non-HS       0.88      0.92      0.90      2681
          HS       0.86      0.80      0.83      1687

    accuracy                           0.87      4368
   macro avg       0.87      0.86      0.87      4368
weighted avg       0.87      0.87      0.87      4368


bert / sbert / example / Vicomtech


              precision    recall  f1-score   support

      Non-HS       0.81      0.82      0.82       239
          HS       0.82      0.80      0.81       239

    accuracy                           0.81       478
   macro avg       0.81      0.81      0.81       478
weighted avg       0.81      0.81      0.81       478


bert / sbert / knowledge / IHC


              precision    recall  f1-score   support

      Non-HS       0.82      0.86      0.84      1330
          HS       0.71      0.65      0.68       698

    accuracy                           0.79      2028
   macro avg       0.77      0.76      0.76      2028
weighted avg       0.79      0.79      0.79      2028


bert / sbert / knowledge / ISHate


...

              precision    recall  f1-score   support

      Non-HS       0.91      0.90      0.91      2681
          HS       0.85      0.85      0.85      1687

    accuracy                           0.88      4368
   macro avg       0.88      0.88      0.88      4368
weighted avg       0.88      0.88      0.88      4368


roberta / sbert / knowledge / Vicomtech


              precision    recall  f1-score   support

      Non-HS       0.59      0.96      0.73       239
          HS       0.90      0.33      0.48       239

    accuracy                           0.65       478
   macro avg       0.74      0.65      0.61       478
weighted avg       0.74      0.65      0.61       478


roberta / sbert / full / IHC


              precision    recall  f1-score   support

      Non-HS       0.83      0.88      0.86      1330
          HS       0.75      0.66      0.70       698

    accuracy                           0.81      2028
   macro avg       0.79      0.77      0.78      2028
weighted avg       0.80      0.81      0.80      2028


roberta / sbert / full / ISHate


              precision    recall  f1-score   support

      Non-HS       0.87      0.94      0.91      2681
          HS       0.90      0.78      0.83      1687

    accuracy                           0.88      4368
   macro avg       0.88      0.86      0.87      4368
weighted avg       0.88      0.88      0.88      4368


roberta / sbert / full / Vicomtech


              precision    recall  f1-score   support

      Non-HS       0.64      0.96      0.77       239
          HS       0.92      0.46      0.61       239

    accuracy                           0.71       478
   macro avg       0.78      0.71      0.69       478
weighted avg       0.78      0.71      0.69       478



## 7. Results

One styled table per dataset.

In [7]:
# Build and display one table per dataset
for ds_name in SELECTED_DATASETS:
    ds_results = {
        k: v for k, v in results.items() if v['dataset'] == ds_name
    }
    if not ds_results:
        print(f'No results for {ds_name}\n')
        continue

    rows = {}
    for label, vals in ds_results.items():
        row_key = f"{vals['model']} / sbert / {vals['index_type']}"
        rows[row_key] = {
            'Macro F1':        vals['macro_f1'],
            'Macro Precision': vals['macro_p'],
            'Macro Recall':    vals['macro_r'],
        }

    df = pd.DataFrame(rows).T
    df.index.name = 'Model / Index'

    display(
        df.style
        .format('{:.3f}')
        .highlight_max(axis=0, props='font-weight: bold; background-color: #d4f1d4')
        .set_caption(f'{ds_name} — RAG weights, plain-text inference (no retrieval)')
    )

,Macro F1,Macro Precision,Macro Recall
Model / Index,,,
bert / sbert / example,0.750,0.765,0.741
bert / sbert / knowledge,0.762,0.769,0.757
bert / sbert / full,0.750,0.766,0.740
roberta / sbert / example,0.785,0.805,0.774
roberta / sbert / knowledge,0.768,0.802,0.754
roberta / sbert / full,0.779,0.790,0.772


,Macro F1,Macro Precision,Macro Recall
Model / Index,,,
bert / sbert / example,0.866,0.872,0.861
bert / sbert / knowledge,0.870,0.870,0.870
bert / sbert / full,0.868,0.873,0.865
roberta / sbert / example,0.866,0.880,0.857
roberta / sbert / knowledge,0.878,0.878,0.879
roberta / sbert / full,0.870,0.883,0.861


,Macro F1,Macro Precision,Macro Recall
Model / Index,,,
bert / sbert / example,0.814,0.814,0.814
bert / sbert / knowledge,0.814,0.815,0.814
bert / sbert / full,0.810,0.810,0.810
roberta / sbert / example,0.543,0.755,0.609
roberta / sbert / knowledge,0.607,0.744,0.646
roberta / sbert / full,0.687,0.777,0.707
